In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

dataset_name = ["MTBench", "HumanEval", "GSM8K", "XSUM", "CNN/DM"]

def simplify_dataset_name(name):
    # 把传入的 name 也去掉所有非字母字符并转为小写
    # "eval/eval_mt_bench_noeval.py" -> "evalevalmtbenchnoevalpy"
    clean_name = "".join(filter(str.isalpha, name)).lower()
    
    for d in dataset_name:
        # "MTBench" -> "mtbench"
        clean_d = "".join(filter(str.isalpha, d)).lower()
        if clean_d in clean_name:
            return d
    return name

method_mapping = {
    "adaptive_tridecoding": "CEE-SD",
    "tridecoding": "TriDecoding",
    "dist_split_spec": "DSSD",
    "dist_spec": "DSD",
    "large": "AR",
    "cee_dsd": "CEE-DSD",
    "cee_dssd": "CEE-DSSD",
    "cee_cuhlm": "CEE-CUHLM",
    
}

simplify_method_name = lambda method: method_mapping.get(method, method)

def read_experiment_data(file_path):
    """读取实验数据文件"""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def extract_experiment_info(experiments):
    """提取实验信息"""
    results = []

    for exp in experiments:
        config = exp.get("config", {})
        result = exp.get("result", {})

        # 提取所需的参数
        eval_dataset = config.get("eval_dataset", "N/A")
        edge_cloud_bandwidth = config.get("edge_cloud_bandwidth", 0)
        edge_end_bandwidth = config.get("edge_end_bandwidth", 0)
        eval_mode = result.get("eval_mode", "")
        accuracy = result.get("accuracy", "N/A")  # 提取 accuracy
        target_forward_times = result.get("target_forward_times", 0)
        generated_tokens = result.get("generated_tokens", 0)
        wall_time = result.get("wall_time", 0)
        communication_time = result.get("communication_time", 0)
        queuing_time = result.get("queuing_time", 0)
        edge_cloud_data_bytes = result.get("edge_cloud_data_bytes", 0)
        throughput = result.get("throughput", 0)
        comm_energy = result.get("comm_energy", 0)

        
        # 提取模型信息
        little_model = result.get("little_model") or config.get("little_model", "N/A")
        draft_model = result.get("draft_model") or config.get("draft_model", "N/A")
        target_model = result.get("target_model") or config.get("target_model", "N/A")
        
        # 为了美观，可以去掉路径前缀
        def simplify_model_name(name):
            if isinstance(name, str) and "/" in name:
                return name.split("/")[-1]
            return name
            
        little_model = simplify_model_name(little_model)
        draft_model = simplify_model_name(draft_model)
        target_model = simplify_model_name(target_model)

        # 提取接受率相关数据
        little_accepted_tokens = result.get("little_accepted_tokens", 0)
        little_generated_tokens = result.get("little_generated_tokens", 0)
        draft_accepted_tokens = result.get("draft_accepted_tokens", 0)
        draft_generated_tokens = result.get("draft_generated_tokens", 0)

        # 计算接受率
        draft_acc = (
            round(little_accepted_tokens / little_generated_tokens * 100, 2)
            if little_generated_tokens > 0
            else "N/A"
        )
        target_acc = (
            round(draft_accepted_tokens / draft_generated_tokens * 100, 2)
            if draft_generated_tokens > 0
            else "N/A"
        )

        # 兼容传统 Speculation
        if target_acc == "N/A" and "acceptance_rate" in result:
            target_acc = round(float(result["acceptance_rate"]) * 100, 2)

        results.append(
            {
                "数据集": simplify_dataset_name(eval_dataset),
                "Edge-Cloud 带宽 (Mbps)": edge_cloud_bandwidth,
                "Edge-End 带宽 (Mbps)": edge_end_bandwidth,
                "方法": simplify_method_name(eval_mode),
                "Little-Model": little_model,
                "Draft-Model": draft_model,
                "Target-Model": target_model,
                "准确率": accuracy,  # 添加到结果中
                "生成 Token 数": generated_tokens,
                "目标模型 Forward 次数": target_forward_times,
                "目标模型 Forward/Token": round(target_forward_times / generated_tokens, 4) if generated_tokens > 0 else None,
                "Draft 接受率 (%)": draft_acc,
                "Target 接受率 (%)": target_acc,
                "挂钟时间 (s)": round(wall_time, 2),
                "通信时间 (s)": round(communication_time, 2),
                "排队时间 (s)": round(queuing_time, 2),
                "Edge-Cloud 数据传输量 (Mbytes)": round(edge_cloud_data_bytes / (1024 * 1024), 2),
                "Token 输出速率 (tokens/s)": round(throughput, 2),
                # "通信能量": comm_energy,
                "Gamma1:": config.get("gamma1", "N/A"),
            }
        )

    return results

def process_data(experiment_data):
    """处理实验数据，计算相对算力消耗并转换为DataFrame"""
    df = pd.DataFrame(experiment_data)

    # 找到 large 方法的基准值
    # 假设每个数据集都有一个 large 实验作为基准
    large_df = df[df['方法'] == 'large'][['数据集', '目标模型 Forward 次数']].drop_duplicates(subset=['数据集'])
    large_df = large_df.rename(columns={'目标模型 Forward 次数': 'baseline_forwards'})

    # 合并基准值
    df = pd.merge(df, large_df, on='数据集', how='left')

    # 计算相对 large 算力消耗 (%)
    # df['相对 large 算力消耗 (%)'] = df.apply(
    #     lambda row: round((row['目标模型 Forward 次数'] / row['baseline_forwards'] * 100), 2)
    #     if pd.notnull(row['baseline_forwards']) and row['baseline_forwards'] > 0
    #     else None,
    #     axis=1
    # )
    
    # Ratio used to compare cloud-model computation after normalizing by output length.
    df['目标模型 Forward/Token'] = df.apply(
        lambda row: round(row['目标模型 Forward 次数'] / row['生成 Token 数'], 4) if row['生成 Token 数'] > 0 else None,
        axis=1
    )

    df['每Token 平均通信时间 (s)'] = df.apply(
        lambda row: round(row['通信时间 (s)'] / row['生成 Token 数'], 4) if row['生成 Token 数'] > 0 else None,
        axis=1
    )

    df['每token平均WAN通信数据量 (bytes)'] = df.apply(
        lambda row: round((row['Edge-Cloud 数据传输量 (Mbytes)'] * 1024 * 1024) / row['生成 Token 数'], 2) if row['生成 Token 数'] > 0 else None,
        axis=1
    )   

    # 清理临时列
    if 'baseline_forwards' in df.columns:
        del df['baseline_forwards']

    if 'Edge-Cloud 数据传输量 (Mbytes)' in df.columns:
        del df['Edge-Cloud 数据传输量 (Mbytes)']
    
    return df

def generate_markdown_table(df):
    """生成 markdown 表格"""
    # 按方法分组，然后按Edge-Cloud带宽降序排序
    df_sorted = df.sort_values(
        ["方法", "Edge-Cloud 带宽 (Mbps)"], ascending=[True, False]
    )

    # 生成 markdown 表格
    markdown_table = df_sorted.to_markdown(index=False, tablefmt="github")
    return markdown_table


def get_pandas_table(df):
    """生成 pandas DataFrame 表格"""
    # 按方法分组，然后按Edge-Cloud带宽降序排序
    df_sorted = df.sort_values(
        ["方法", "Edge-Cloud 带宽 (Mbps)"], ascending=[True, False]
    )
    return df_sorted


def generate_grouped_tables(df):
    """按方法分组生成多个表格"""
    # 按方法分组
    grouped = df.groupby("方法")

    all_tables = []

    for method, group_df in grouped:
        # 按Edge-Cloud带宽降序排序
        group_df_sorted = group_df.sort_values(
            "Edge-Cloud 带宽 (Mbps)", ascending=False
        )

        # 生成表格
        table = group_df_sorted.to_markdown(index=False, tablefmt="github")

        all_tables.append(f"## 方法: {method}\n\n{table}\n")

    return "\n".join(all_tables)


def generate_dataset_grouped_tables(df):
    """按数据集分组生成多个表格"""
    # 按数据集分组
    grouped = df.groupby("数据集")

    all_tables = []

    for dataset, group_df in grouped:
        # 排序：按方法排序，带宽降序
        group_df_sorted = group_df.sort_values(
            ["方法", "Edge-Cloud 带宽 (Mbps)"], ascending=[True, False]
        )

        # 生成表格
        table = group_df_sorted.to_markdown(index=False, tablefmt="github")

        all_tables.append(f"## 数据集: {dataset}\n\n{table}\n")

    return "\n".join(all_tables)


def generate_per_dataset_avg_table(df):
    """按数据集生成各方法平均对比表（跨模型系列平均）"""
    all_tables = []
    for dataset in sorted(df["数据集"].unique()):
        ds_df = df[df["数据集"] == dataset]
        avg_table = ds_df.groupby("方法").mean(numeric_only=True).round(2)
        avg_table = avg_table.sort_values("Token 输出速率 (tokens/s)", ascending=False)
        table = avg_table.to_markdown(tablefmt="github")
        all_tables.append(f"## 数据集: {dataset} (跨系列平均)\n\n{table}\n")
    return "\n".join(all_tables)


def jupyter_display_per_dataset_avg(df):
    """在 Jupyter 中显示各数据集平均方法对比"""
    for dataset in sorted(df["数据集"].unique()):
        display(Markdown(f"## 数据集: {dataset} (跨模型系列平均)"))
        ds_df = df[df["数据集"] == dataset]
        avg_table = ds_df.groupby("方法").mean(numeric_only=True).round(4)
        avg_table = avg_table.sort_values("Token 输出速率 (tokens/s)", ascending=False)

        display(avg_table)



def jupyter_display(df):
    """
    一个专门为 Jupyter Notebook 设计的主函数，
    用于读取、处理数据并以美观的表格形式显示结果。
    """
    # --- 1. 显示统一排序的表格 ---
    # 使用 display(Markdown(...)) 来渲染标题
    display(Markdown("# 实验结果对比表（按方法和带宽排序）"))

    # 对 DataFrame 进行排序
    df_sorted = df.sort_values(
        ["方法", "Edge-Cloud 带宽 (Mbps)"], ascending=[True, False]
    )

    # 直接显示 DataFrame 对象，Jupyter 会将其渲染为 HTML 表格
    display(df_sorted)
    
    # --- 2. 显示按方法分组的表格 ---
    display(Markdown("# 实验结果对比表（按方法分组）"))

    # 按方法进行分组
    grouped = df.groupby("方法")
    # 遍历每个分组并分别显示
    for method, group_df in grouped:
        # 显示分组标题
        display(Markdown(f"## 方法: {method}"))

        # 对当前分组的 DataFrame 进行排序
        group_df_sorted = group_df.sort_values(
            "Edge-Cloud 带宽 (Mbps)", ascending=False
        )

        # 显示排序后的分组 DataFrame
        display(group_df_sorted)

    # --- 4. 显示各数据集方法平均对比表（跨模型系列平均） ---
    display(Markdown("# 各数据集方法平均对比 (跨模型系列)"))
    jupyter_display_per_dataset_avg(df)

    # --- 3. 显示按数据集分组的表格 ---
    display(Markdown("# 实验结果对比表（按数据集分组）"))

    # 按数据集进行分组
    grouped_ds = df.groupby("数据集")
    for dataset, group_df in grouped_ds:
        # 显示分组标题
        display(Markdown(f"## 数据集: {dataset}"))

        # 排序：按方法排序，带宽降序
        group_df_sorted = group_df.sort_values(
            ["方法", "Edge-Cloud 带宽 (Mbps)"], ascending=[True, False]
        )

        # 显示排序后的分组 DataFrame
        display(group_df_sorted)


def main(specified_file=None):
    # 读取实验数据
    import glob
    import os
    
    file_path = None
    candidate_dirs = [Path('experiment_results'), Path('../experiment_results'), Path('.')]
    results_dir = next((path for path in candidate_dirs if path.exists()), Path('experiment_results'))
    
    # 1. 优先使用用户指定的文件
    if specified_file:
        if os.path.exists(specified_file):
            file_path = specified_file
        else:
            print(f"指定的文件不存在: {specified_file}")
            return

    # 2. 如果没有指定文件，则寻找最新的 experiment_summary_*.json
    if not file_path:
        json_files = glob.glob(str(results_dir / "experiment_summary_*.json"))
        if json_files:
            file_path = max(json_files, key=os.path.getmtime)
        else:
            # 3. 最后的兜底
            target_file = results_dir / "experiment_summary_20260127_053240.json"
            if os.path.exists(target_file):
                file_path = str(target_file)
            else:
                print("找不到任何实验总结 JSON 文件")
                return
            
    print(f"处理文件: {file_path}")

    experiments = read_experiment_data(file_path)
    experiment_data = extract_experiment_info(experiments)

    # process data to dataframe and add new metrics
    df = process_data(experiment_data)

    unified_table = generate_markdown_table(df)
    method_grouped_tables = generate_grouped_tables(df)
    dataset_grouped_tables = generate_dataset_grouped_tables(df)

    jupyter_display(df)

    # 保存到文件
    with open("experiment_results_table.md", "w", encoding="utf-8") as f:
        f.write("# 实验结果对比表\n\n")
        f.write("## 1. 统一对比表（按方法和带宽排序）\n\n")
        f.write(unified_table)
        f.write("\n\n" + "=" * 80 + "\n\n")
        f.write("## 2. 按方法分组表格\n\n")
        f.write(method_grouped_tables)
        f.write("\n\n" + "=" * 80 + "\n\n")
        f.write("## 3. 按数据集分组表格\n\n")
        f.write("\n\n" + "=" * 80 + "\n\n")
        f.write("## 4. 各数据集方法平均对比表\n\n")
        per_dataset_avg = generate_per_dataset_avg_table(df)
        f.write(per_dataset_avg)

    print("\n表格已保存到 experiment_results_table.md 文件中")


if __name__ == "__main__":
    # 如果想指定文件，请修改下一行，例如: main("experiment_results/experiment_summary_20260127_053240.json")
    main()

处理文件: ../experiment_results/experiment_summary_fluid_gamma_20261010.json


# 实验结果对比表（按方法和带宽排序）

,数据集,Edge-Cloud 带宽 (Mbps),Edge-End 带宽 (Mbps),方法,Little-Model,Draft-Model,Target-Model,准确率,生成 Token 数,目标模型 Forward 次数,目标模型 Forward/Token,Draft 接受率 (%),Target 接受率 (%),挂钟时间 (s),通信时间 (s),排队时间 (s),Token 输出速率 (tokens/s),Gamma1:,每Token 平均通信时间 (s),每token平均WAN通信数据量 (bytes)
12,MTBench,46.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,NaN,20283,4769,0.2351,66.15,77.64,824.06,242.01,238.45,24.61,5,0.0119,158.71
27,GSM8K,46.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,0.2375,10240,2922,0.2854,55.89,73.27,499.86,149.26,146.10,20.49,5,0.0146,314.37
42,HumanEval,46.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,0.0000,10240,1787,0.1745,44.57,90.35,332.56,91.08,89.35,30.79,5,0.0089,101.38
57,MTBench,46.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,NaN,17938,6444,0.3592,58.02,49.13,1049.80,330.24,322.20,17.09,5,0.0184,634.83
72,GSM8K,46.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,0.1875,10197,2910,0.2854,76.79,67.21,474.62,150.65,145.50,21.48,5,0.0148,821.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,GSM8K,5.0,563,uncertainty_decoding,Qwen1.5-0.5B-Chat,Qwen1.5-0.5B-Chat,Qwen1.5-7B-Chat,0.1500,10236,4837,0.4725,N/A,53.79,690.81,259.19,241.85,14.82,3,0.0253,481.47
83,HumanEval,5.0,563,uncertainty_decoding,Qwen1.5-0.5B-Chat,Qwen1.5-0.5B-Chat,Qwen1.5-7B-Chat,0.0000,10225,3550,0.3472,N/A,37.91,521.65,192.59,177.50,19.60,3,0.0188,357.90
98,MTBench,5.0,563,uncertainty_decoding,Qwen3-0.6B,Qwen3-0.6B,Qwen3-14B,NaN,20539,9036,0.4399,N/A,37.83,1544.43,487.84,451.80,13.30,3,0.0238,445.18
113,GSM8K,5.0,563,uncertainty_decoding,Qwen3-0.6B,Qwen3-0.6B,Qwen3-14B,0.0500,10258,3281,0.3198,N/A,27.54,591.24,180.58,164.05,17.35,3,0.0176,333.24


# 实验结果对比表（按方法分组）

## 方法: CEE-SD

...

## 数据集: HumanEval

,数据集,Edge-Cloud 带宽 (Mbps),Edge-End 带宽 (Mbps),方法,Little-Model,Draft-Model,Target-Model,准确率,生成 Token 数,目标模型 Forward 次数,目标模型 Forward/Token,Draft 接受率 (%),Target 接受率 (%),挂钟时间 (s),通信时间 (s),排队时间 (s),Token 输出速率 (tokens/s),Gamma1:,每Token 平均通信时间 (s),每token平均WAN通信数据量 (bytes)
42,HumanEval,46.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,0.0,10240,1787,0.1745,44.57,90.35,332.56,91.08,89.35,30.79,5,0.0089,101.38
87,HumanEval,46.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,0.0,10208,1820,0.1783,77.9,77.27,326.20,92.22,91.00,31.29,5,0.0090,50.33
43,HumanEval,10.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,0.0,10240,1787,0.1745,44.57,90.35,336.65,94.91,89.35,30.42,5,0.0093,101.38
88,HumanEval,10.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,0.0,10208,1819,0.1782,77.9,77.21,326.61,94.35,90.95,31.25,5,0.0092,49.31
44,HumanEval,5.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,0.0,10240,1787,0.1745,44.57,90.35,341.23,99.38,89.35,30.01,5,0.0097,101.38
89,HumanEval,5.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,0.0,10208,1819,0.1782,77.9,77.21,328.94,97.03,90.95,31.03,5,0.0095,49.31
30,HumanEval,46.0,563,DSD,llama-68m,llama-68m,Llama-2-13b-hf,0.0,10240,4912,0.4797,N/A,36.79,982.81,481.41,245.60,10.42,3,0.0470,113674.24
75,HumanEval,46.0,563,DSD,Qwen1.5-0.5B-Chat,Qwen1.5-0.5B-Chat,Qwen1.5-7B-Chat,0.0,10208,3114,0.3051,N/A,76.98,1008.16,710.82,155.70,10.13,3,0.0696,302622.24
120,HumanEval,46.0,563,DSD,Qwen3-0.6B,Qwen3-0.6B,Qwen3-14B,0.0,10240,3538,0.3455,N/A,64.18,1253.68,810.84,176.90,8.17,3,0.0792,360761.34
31,HumanEval,10.0,563,DSD,llama-68m,llama-68m,Llama-2-13b-hf,0.0,10240,6546,0.6393,N/A,56.90,1464.94,809.06,327.30,6.99,3,0.0790,58100.74


## 数据集: MTBench

,数据集,Edge-Cloud 带宽 (Mbps),Edge-End 带宽 (Mbps),方法,Little-Model,Draft-Model,Target-Model,准确率,生成 Token 数,目标模型 Forward 次数,目标模型 Forward/Token,Draft 接受率 (%),Target 接受率 (%),挂钟时间 (s),通信时间 (s),排队时间 (s),Token 输出速率 (tokens/s),Gamma1:,每Token 平均通信时间 (s),每token平均WAN通信数据量 (bytes)
12,MTBench,46.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,NaN,20283,4769,0.2351,66.15,77.64,824.06,242.01,238.45,24.61,5,0.0119,158.71
57,MTBench,46.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,NaN,17938,6444,0.3592,58.02,49.13,1049.80,330.24,322.20,17.09,5,0.0184,634.83
102,MTBench,46.0,563,CEE-SD,Qwen3-0.6B,Qwen3-1.7B,Qwen3-14B,NaN,20480,5724,0.2795,64.58,63.11,1150.05,290.18,286.20,17.81,5,0.0142,173.57
13,MTBench,10.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,NaN,20283,4769,0.2351,66.14,77.68,831.02,248.87,238.45,24.41,5,0.0123,158.19
58,MTBench,10.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,NaN,17938,6462,0.3602,57.89,48.85,1073.75,352.04,323.10,16.71,5,0.0196,774.54
103,MTBench,10.0,563,CEE-SD,Qwen3-0.6B,Qwen3-1.7B,Qwen3-14B,NaN,20480,5724,0.2795,64.58,63.11,1157.93,297.52,286.20,17.69,5,0.0145,173.57
14,MTBench,5.0,563,CEE-SD,llama-68m,tiny-llama-1.1b,Llama-2-13b-hf,NaN,20283,4769,0.2351,66.12,77.68,838.70,256.82,238.45,24.18,5,0.0127,158.19
59,MTBench,5.0,563,CEE-SD,Qwen1.5-0.5B-Chat,Qwen1.5-1.8B-Chat,Qwen1.5-7B-Chat,NaN,17938,6462,0.3602,57.89,48.85,1095.56,374.16,323.10,16.37,5,0.0209,786.81
104,MTBench,5.0,563,CEE-SD,Qwen3-0.6B,Qwen3-1.7B,Qwen3-14B,NaN,20480,5724,0.2795,64.58,63.11,1166.31,306.04,286.20,17.56,5,0.0149,173.57
0,MTBench,46.0,563,DSD,llama-68m,llama-68m,Llama-2-13b-hf,NaN,20480,7883,0.3849,N/A,54.02,1565.61,776.46,394.15,13.08,3,0.0379,87455.74



表格已保存到 experiment_results_table.md 文件中
